# Config

In [1]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

In [2]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [3]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean, detect_language
import time
import json
import numpy as np

# Preparación de texto

In [52]:
# 1) Cargar datos
path = "/tmp/desafios/new_data"
#Cargar titulo, keys, etc.
filePATH = os.path.join(path, "new_data.xlsx")
df_data = pd.read_excel(filePATH, usecols=["Código VRID", "Título", "Keywords", "Resumen"])
#Cargar desafíos
filePATH = os.path.join(path, "Datos_DP.xlsx")
df_labels = pd.read_excel(filePATH, usecols=["Código VRID", "Desafío País"])

In [59]:
print(df_data.shape, df_labels.shape)
print(np.unique(df_data["Código VRID"].astype(str)).shape)
print(np.unique(df_labels["Código VRID"].astype(str)).shape)

(1456, 4) (1404, 2)
(1071,)
(1404,)


In [64]:
# Merge en base a "Código VRID"
df_merged = pd.merge(
    df_data,
    df_labels,
    on="Código VRID",       # columna clave
    how="inner"             # inner = solo los que coinciden en ambos
)
#Eliminar dupliados
df_merged = df_merged.drop_duplicates(subset=["Código VRID"], keep="first")
print(df_merged.shape)                                                                                                                                
#print(np.unique(df_merged["Código VRID"].astype(str)).shape)
print(df_merged["Desafío País"].value_counts())

#save dataframe
savepath=os.path.join(path, "data_concatenada.csv")
df_merged.to_csv(savepath, index=False, encoding="utf-8-sig")

(1062, 5)
0        320
3        269
2.4      144
4        117
2        110
1         60
1,2,3     33
3.4        4
Name: Desafío País, dtype: int64


# 1) Preprocesamiento de los datos


In [73]:
# 1) Cargar datos
path = "/tmp/desafios/new_data"
filePATH = os.path.join(path, "data_concatenada.csv")
df = pd.read_csv(filePATH)

# 2) Guardar qué secuencias de palabras del resumen serán eliminadas al aplicar get_expressions_to_delete()
list_texts = df["Resumen"].to_list()
df_deleted = check_deleted_expressions(list_texts)

# 3) Preprocesar los datos
#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    #"Facultad del Proyecto": "Facultad_del_Proyecto_trad",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)
savepath=os.path.join(path, "data_clean.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

# 2) Traducción del texto

In [74]:
#Crear columna de registro de idioma: 
# True: Texto en español, False: Texto en inglés
df["Español"]=detect_language(df["Resumen_trad"])

In [75]:
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    #"Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    #"Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
#####Estoy trabajando en mejorar esta parte para que sea más rápida con paralelización por batches
start = time.time()
for src, dst in cols.items():
    df[dst] = trans.translate_parallel(df[src].to_list(), batch_size=8)
end = time.time()


#3.Guardado de resultados
savepath=os.path.join(path, "data_translated.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

print(f"Tiempo total de traducción: {end - start:.2f} segundos")

/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Usando dispositivo: cuda


Traduciendo: 100%|██████████| 47/47 [00:10<00:00,  4.34batch/s]

Tiempo total de traducción: 712.09 segundos


In [76]:
df.head()

,Código VRID,Título,Keywords,Resumen,Desafío País,Titulo_trad,Resumen_trad,keywords_trad,Español
0,15130015,CENTRO DE RECURSOS HIDRICOS PARA AGRICULTURA Y...,NaN,PROPOSAL DESCRIPTION_x000D_\nWATER AVAILABILIT...,2.4,hydric resource centre for agriculture and min...,proposal description \nwater availability and ...,,False
1,IC120019,INSTITUTO MILENIO DE OCEANOGRAFÍA INTEGRATIVA ...,NaN,EL INSTITUTO MILENIO DE OCEANOGRAFÍA (IMO) APU...,2.4,millennium institute of integrative oceanograp...,the millenium institute of oceanography (imo) ...,,True
2,217.173.049-1.0,PATRONES DE CRIANZA Y SOCIALIZACIÓN DE GÉNERO ...,NaN,OBJETIVOS GENERALES: _x000D_\nDESCRIBIR LOS PR...,4,patterns of gender upbringing and socializatio...,general objectives: to describe the processes ...,,True
3,11170959,PRE-AND POST-NATAL EXPRESSION OF SLIT2 AND ITS...,"NEUROBLAST MIGRATION, SLIT2, LATERAL VENTRICLE...",IN THE SUBVENTRICULAR ZONE (SVZ) OF THE ADULT ...,4,pre-and post-natal expression of slit2 and its...,in the subventricular zone (svz) of the adult ...,"neuroblast migration, slit2, lateral ventricle...",False
4,218.201.002-1.0,ADAPTACIÓN CULTURAL Y VALIDACIÓN DE LA ESCALA ...,"ESTILO DE VIDA, ADOLESCENTES _x000D_\n",PARA EVALUAR LOS COMPORTAMIENTOS RELACIONADOS ...,0,cultural adaptation and validation of the life...,In order to evaluate the behaviors related to ...,"lifestyle, teens",True


# 3) Split dataset

In [79]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from utils.dataset import to_serializable
import json

path = "/tmp/desafios/new_data"
filepath=os.path.join(path, "data_translated.csv")
df = pd.read_csv(filepath)



In [80]:
le = LabelEncoder()
df["labels"] = le.fit_transform(df["Desafío País"].astype(str))

# Convertir a arrays
ids = df["Código VRID"].to_numpy()
labels = df["labels"].to_numpy()

# Train/Test split (ids y labels en paralelo)
idx_train, idx_test, y_train, y_test = train_test_split(
    ids,
    labels,
    test_size=0.2,
    random_state=7,
    stratify=labels
)

# Crear folds sobre train
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)

folds = []
for fold, (train_pos, val_pos) in enumerate(skf.split(idx_train, y_train)):
    train_ids = idx_train[train_pos]   # array de IDs
    val_ids = idx_train[val_pos]       # array de IDs
    folds.append(val_ids)

print("Test size:", len(idx_test))
print("Fold 0 - Val size:", len(folds[0]))

#Guardar index en diccionario
dataset_index = {
    "Train": idx_train,
    "Test": idx_test,
    "kfolds": folds 
}
filepath=os.path.join(path, "new_data_ids_3folds.json")

# Guardar
with open(filepath, "w", encoding="utf-8") as f:
    json.dump(dataset_index, f, default=to_serializable, indent=2, ensure_ascii=False)

Test size: 213
Fold 0 - Val size: 283


# 4) TF-ID feature extractor 

## Train

In [12]:
import numpy as np

def binarize_labels(y_train, y_test, positive_labels):
    """
    Binariza los vectores de labels según una lista de condiciones positivas.

    Parámetros:
        y_train (array-like): etiquetas de entrenamiento
        y_test (array-like): etiquetas de prueba
        positive_labels (list): lista de valores que deben considerarse como 1

    Retorna:
        y_train_bin, y_test_bin (arrays numpy con 0 y 1)
    """

    # Normalizar entradas a string y quitar espacios
    y_train = np.char.strip(np.array(y_train).astype(str))
    y_test = np.char.strip(np.array(y_test).astype(str))

    # Crear condición general
    cond_train = np.isin(y_train, positive_labels)
    cond_test = np.isin(y_test, positive_labels)

    # Aplicar binarización
    y_train_bin = np.where(cond_train, 1, 0).astype(int)
    y_test_bin = np.where(cond_test, 1, 0).astype(int)

    return y_train_bin, y_test_bin


In [13]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/desafios/new_data"
filepath=os.path.join(path, "new_data_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "data_translated.csv")
df = pd.read_csv(filepath)


In [30]:
df["Desafío País"].shape

(1062,)

In [14]:
df["Desafío País"].value_counts()

0        320
3        269
2.4      144
4        117
2        110
1         60
1,2,3     33
3.4        4
Name: Desafío País, dtype: int64

In [46]:
from sklearn.preprocessing import LabelEncoder
from models.TIFD import gen_TFID_vectors
from utils.dataset import gen_dataset_select_cols
import numpy as np

#Lectura de codigos 
codes_test = dataset_index["Test"]
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                test_col="Desafío País")

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols, 
                                                     test_col="Desafío País")
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
positive_labels = ["4", "2.4", "3.4"]
y_train, y_test = binarize_labels(y_train, y_test, positive_labels)
#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)
print(X_train.shape, X_test.shape)

(849, 17069) (213, 17069)


In [47]:
from collections import Counter
print("Train label distribution:", Counter(y_train))
print("Test label distribution:", Counter(y_test))

Train label distribution: Counter({0: 637, 1: 212})
Test label distribution: Counter({0: 160, 1: 53})


In [48]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model#, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    'RandomForestClassifier',
    'XGBClassifier',
    'SVC',
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring='recall'
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({0: 637, 1: 212})
📊 test: Counter({0: 160, 1: 53})
(606,)
LogisticRegression
Compute sw


/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [1.0, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', 'lbfgs'] before, using random point [10, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', '

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw


/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point ['logloss', 0.01, 3, 40] before, using random point ['logloss', 0.1337067770229036, 5, 224]
  warnings.warn(


SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.38, 'std_test_score': 0.17}
RandomForestClassifier: {'mean_test_score': 0.22, 'std_test_score': 0.13}
XGBClassifier: {'mean_test_score': 0.48, 'std_test_score': 0.08}
SVC: {'mean_test_score': 0.36, 'std_test_score': 0.43}


In [49]:
from utils.mlflow import eval_model
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results, preds=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.7417840375586855, 'f1_macro': 0.6094088620678157, 'cm': array([[141,  19],
       [ 36,  17]]), 'precision': 0.4722222222222222, 'recall': 0.32075471698113206, 'f1_es': 0.7737798452800124, 'f1_en': 0.6459737254101151, 'cm_es': array([[96,  9],
       [18,  7]]), 'cm_en': array([[45, 10],
       [18, 10]])}
RandomForestClassifier
{'accuracy': 0.7089201877934272, 'f1_macro': 0.5632936507936508, 'cm': array([[137,  23],
       [ 39,  14]]), 'precision': 0.3783783783783784, 'recall': 0.2641509433962264, 'f1_es': 0.7673611111111112, 'f1_en': 0.5621055651176134, 'cm_es': array([[99,  6],
       [20,  5]]), 'cm_en': array([[38, 17],
       [19,  9]])}
XGBClassifier
{'accuracy': 0.7605633802816901, 'f1_macro': 0.6378154902810655, 'cm': array([[143,  17],
       [ 34,  19]]), 'precision': 0.5277777777777778, 'recall': 0.3584905660377358, 'f1_es': 0.7795454545454545, 'f1_en': 0.6862909513511922, 'cm_es': array([[97,  8],
       [18,  7]]), 'cm_en': array([[46,  

## Save

In [50]:
from utils.mlflow import mlflow_ckeckpoint

# Guardar en MLflow
exp_info = {
    'exp_name': "Bayesiansearchcv_desafios_all_results2",
}

extra_parms = {
    "vectorization_model": "TF-IDF",
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "cols": cols,
    "positive_labels": positive_labels,
}

extra_artifacts = {
    "dataset_data_splits": dataset_index,
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, X_test, y_test, df_test, save_preds=True, lang_es=lang_es, extra_parms=extra_parms, extra_artifacts=extra_artifacts, mode="server", mode_classification="binary")

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/09/17 22:31:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/20/runs/8e6c464cb59b4f318c1fbe17079a3b12
🧪 View experiment at: http://mlflow-server:5000/#/experiments/20
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/09/17 22:31:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/20/runs/b35d0bd9c1d44155a8464c7f47773e81
🧪 View experiment at: http://mlflow-server:5000/#/experiments/20
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/17 22:31:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/20/runs/11109a1efdfa47089e4822d30a52dd3b
🧪 View experiment at: http://mlflow-server:5000/#/experiments/20
📝 Registrando modelo en MLflow: SVC


2025/09/17 22:31:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/20/runs/a3972ab13e4943e2b11a0fd6bd3873f3
🧪 View experiment at: http://mlflow-server:5000/#/experiments/20


# 5) SPECTER model

## Train

In [3]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [4]:
#del gen_dataset
from utils.dataset import gen_dataset
from models.specter import embed_texts
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# 2) Calcular embeddings
# Parámetros modelo
BASE_MODEL = "allenai/specter2_base"
#ADAPTER_NAME = "allenai/specter2"
ADAPTER_NAME="allenai/specter2_classification"
X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)

print(X_train.shape, X_test.shape)

/usr/local/lib/python3.10/dist-packages/torch/_utils.py:830: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
(771, 768) (193, 768)


In [5]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring="f1_macro"
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.64, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.63, 'std_test_score': 0.01}
XGBClassifier: {'mean_test_score': 0.62, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.63, 'std_test_score': 0.01}


In [6]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6476683937823834, 'precision': 0.7087378640776699, 'recall': 0.6576576576576577, 'f1_macro': 0.6434470767224516, 'cm': array([[52, 30],
       [38, 73]]), 'f1_es': 0.6325030804435839, 'f1_en': 0.6575154426904598, 'cm_es': array([[18, 22],
       [20, 55]]), 'cm_en': array([[34,  8],
       [18, 18]])}
RandomForestClassifier
{'accuracy': 0.6683937823834197, 'precision': 0.7079646017699115, 'recall': 0.7207207207207207, 'f1_macro': 0.6596119929453264, 'cm': array([[49, 33],
       [31, 80]]), 'f1_es': 0.6373445535296867, 'f1_en': 0.7038019451812555, 'cm_es': array([[17, 23],
       [18, 57]]), 'cm_en': array([[32, 10],
       [13, 23]])}
XGBClassifier
{'accuracy': 0.6839378238341969, 'precision': 0.7049180327868853, 'recall': 0.7747747747747747, 'f1_macro': 0.6697523072175937, 'cm': array([[46, 36],
       [25, 86]]), 'f1_es': 0.6669747772937873, 'f1_en': 0.6927133512499366, 'cm_es': array([[17, 23],
       [14, 61]]), 'cm_en': array([[29, 13],
       [1

## Save

In [ ]:
exp_info = {
    'exp_name': "Bayesiansearchcv_specter_f1m",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/08/29 20:00:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/1/runs/59d9bf17d81740c9808412615b4f1223
🧪 View experiment at: http://mlflow-server:5000/#/experiments/1
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/08/29 20:00:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/1/runs/43ab81c635d84877871f6c191ab03d9d
🧪 View experiment at: http://mlflow-server:5000/#/experiments/1
📝 Registrando modelo en MLflow: XGBClassifier


2025/08/29 20:00:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/1/runs/3b4c012864f34c639fdfa554a4fce910
🧪 View experiment at: http://mlflow-server:5000/#/experiments/1
📝 Registrando modelo en MLflow: SVC


2025/08/29 20:00:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/1/runs/a4ecef4973ba4ca0bb6d6fa338bb7e92
🧪 View experiment at: http://mlflow-server:5000/#/experiments/1
